In [ ]:
# https://github.com/ngquyn2602/Bai_tap_tri_tue_nhan_tao_AI
import random
import time
from IPython.display import clear_output


# =========================================================
# PEAS - MÁY HÚT BỤI KHÔNG CÓ VẬT CẢN
# =========================================================

PEAS = {
    "P - Performance": "Hút sạch bụi, số bước càng ít càng tốt",
    "E - Environment": "Phòng 3x3, có ô sạch và ô bẩn, không có vật cản",
    "A - Actuators": "SUCK, UP, DOWN, LEFT, RIGHT, STOP",
    "S - Sensors": "Biết vị trí hiện tại, biết ô hiện tại có bụi không, biết số bụi còn lại, biết action hợp lệ"
}

ROWS = 3
COLS = 3

ACTIONS = {
    "UP": (-1, 0),
    "DOWN": (1, 0),
    "LEFT": (0, -1),
    "RIGHT": (0, 1)
}


# =========================================================
# E - ENVIRONMENT
# =========================================================

def create_environment(agent_pos, dirt_cells):
    return {
        "agent_pos": agent_pos,
        "dirt": set(dirt_cells),
        "initial_dirt_count": len(dirt_cells),
        "step": 0
    }


def random_environment(dirt_count):
    all_cells = [(r, c) for r in range(ROWS) for c in range(COLS)]
    agent_pos = (0, 0)
    dirt_cells = set(random.sample(all_cells, dirt_count))

    return create_environment(agent_pos, dirt_cells)


def copy_environment(env):
    return {
        "agent_pos": env["agent_pos"],
        "dirt": set(env["dirt"]),
        "initial_dirt_count": env["initial_dirt_count"],
        "step": env["step"]
    }


def is_inside(row, col):
    return 0 <= row < ROWS and 0 <= col < COLS


# =========================================================
# S - SENSORS
# =========================================================

def is_dirty(env):
    return env["agent_pos"] in env["dirt"]


def get_legal_actions(env):
    row, col = env["agent_pos"]
    legal_actions = []

    for action, (dr, dc) in ACTIONS.items():
        new_row = row + dr
        new_col = col + dc

        if is_inside(new_row, new_col):
            legal_actions.append(action)

    return legal_actions


def sensor(env):
    return {
        "position": env["agent_pos"],
        "dirty_here": is_dirty(env),
        "dirt_left": len(env["dirt"]),
        "legal_actions": get_legal_actions(env)
    }


# =========================================================
# P - PERFORMANCE
# =========================================================

def performance(env):
    return env["initial_dirt_count"] - len(env["dirt"])


# =========================================================
# SIMPLE REFLEX AGENT + RULE
# =========================================================

def simple_vacuum_agent(percept):
    row, col = percept["position"]

    # Rule 1: Nếu ô hiện tại bẩn thì hút
    if percept["dirty_here"]:
        return "SUCK", "IF ô hiện tại bẩn THEN SUCK"

    # Rule 2: Nếu không còn bụi thì dừng
    if percept["dirt_left"] == 0:
        return "STOP", "IF không còn bụi THEN STOP"

    # Rule 3: Nếu ô hiện tại sạch thì đi theo đường cố định
    if row == 0 and col == 0:
        return "RIGHT", "Ô (0,0) sạch THEN đi RIGHT"

    if row == 0 and col == 1:
        return "RIGHT", "Ô (0,1) sạch THEN đi RIGHT"

    if row == 0 and col == 2:
        return "DOWN", "Ô (0,2) sạch THEN đi DOWN"

    if row == 1 and col == 2:
        return "LEFT", "Ô (1,2) sạch THEN đi LEFT"

    if row == 1 and col == 1:
        return "LEFT", "Ô (1,1) sạch THEN đi LEFT"

    if row == 1 and col == 0:
        return "DOWN", "Ô (1,0) sạch THEN đi DOWN"

    if row == 2 and col == 0:
        return "RIGHT", "Ô (2,0) sạch THEN đi RIGHT"

    if row == 2 and col == 1:
        return "RIGHT", "Ô (2,1) sạch THEN đi RIGHT"

    if row == 2 and col == 2:
        return "STOP", "Ô (2,2) sạch THEN STOP"

    return "STOP", "Không có rule phù hợp"


# =========================================================
# A - ACTUATORS
# =========================================================

def apply_action(env, action):
    row, col = env["agent_pos"]

    if action == "SUCK":
        if (row, col) in env["dirt"]:
            env["dirt"].remove((row, col))

    elif action in ACTIONS:
        dr, dc = ACTIONS[action]
        new_row = row + dr
        new_col = col + dc

        if is_inside(new_row, new_col):
            env["agent_pos"] = (new_row, new_col)

    env["step"] += 1


# =========================================================
# TERMINAL OUTPUT
# =========================================================

def line():
    print("=" * 82)


def print_peas():
    line()
    print("PEAS CHO AGENT MÁY HÚT BỤI - KHÔNG CÓ VẬT CẢN")
    line()

    for key, value in PEAS.items():
        print(key + ":")
        print("  -", value)
        print()


def print_agent_path():
    line()
    print("ĐƯỜNG ĐI CỐ ĐỊNH CỦA SIMPLE AGENT")
    line()
    print("(0,0) -> (0,1) -> (0,2)")
    print("                    ↓")
    print("(1,0) <- (1,1) <- (1,2)")
    print("↓")
    print("(2,0) -> (2,1) -> (2,2)")
    print()


def cell_symbol(env, cell):
    if cell == env["agent_pos"] and cell in env["dirt"]:
        return "AD"

    if cell == env["agent_pos"]:
        return "A"

    if cell in env["dirt"]:
        return "D"

    return "."


def print_board(env):
    print("+------+------+------+")
    for r in range(ROWS):
        row_text = "|"
        for c in range(COLS):
            symbol = cell_symbol(env, (r, c))
            row_text += f"  {symbol:^2}  |"
        print(row_text)
        print("+------+------+------+")

    print("A = Agent | D = Dirt | AD = Agent đứng trên bụi | . = Sạch")
    print()


def print_initial_state(initial_env):
    line()
    print("TRẠNG THÁI BAN ĐẦU")
    line()
    print("Step ban đầu   :", initial_env["step"])
    print("Performance    :", f"{performance(initial_env)} / {initial_env['initial_dirt_count']}")
    print("Agent position :", initial_env["agent_pos"])
    print("Số bụi ban đầu :", initial_env["initial_dirt_count"])
    print()

    print_board(initial_env)


def print_current_state(env, mode):
    percept = sensor(env)

    line()
    print("TRẠNG THÁI HIỆN TẠI")
    line()
    print("Mode           :", mode)
    print("Step hiện tại  :", env["step"])
    print("Performance    :", f"{performance(env)} / {env['initial_dirt_count']}")
    print("Agent position :", percept["position"])
    print("Dirty here     :", percept["dirty_here"])
    print("Dirt left      :", percept["dirt_left"])
    print("Legal actions  :", percept["legal_actions"])
    print()

    print_board(env)


def print_agent_decision(rule, action):
    line()
    print("SIMPLE AGENT DECISION")
    line()
    print("Rule   :", rule)
    print("Action :", action)
    print()


def print_history(history):
    line()
    print("LỊCH SỬ ACTION")
    line()

    if len(history) == 0:
        print("Chưa có action nào.")
    else:
        for item in history:
            print(item)

    print()


def print_result(env):
    line()
    if len(env["dirt"]) == 0:
        print("KẾT QUẢ: ĐÃ HÚT SẠCH BỤI")
    else:
        print("KẾT QUẢ: CHƯA HÚT SẠCH BỤI")
    print("Performance:", f"{performance(env)} / {env['initial_dirt_count']}")
    line()


# =========================================================
# RUN - AGENT SIMPLE
# =========================================================

def run_agent_simple(env, max_steps=50, delay=0.4, animate=True):
    history = []
    initial_env = copy_environment(env)

    while env["step"] < max_steps:
        if animate:
            clear_output(wait=True)

        percept = sensor(env)
        action, rule = simple_vacuum_agent(percept)

        print_peas()
        print_agent_path()
        print_initial_state(initial_env)
        print_current_state(env, mode="AGENT SIMPLE")
        print_agent_decision(rule, action)
        print_history(history)

        if action == "STOP":
            print_result(env)
            break

        old_pos = percept["position"]
        old_dirty = percept["dirty_here"]
        old_perf = performance(env)

        apply_action(env, action)

        new_pos = env["agent_pos"]
        new_perf = performance(env)

        history.append(
            f"Step {env['step']:02d} | Action: {action:<5} | "
            f"Position: {old_pos} -> {new_pos} | "
            f"Dirty before: {old_dirty} | "
            f"Performance: {old_perf}/{env['initial_dirt_count']} -> {new_perf}/{env['initial_dirt_count']}"
        )

        if len(env["dirt"]) == 0:
            if animate:
                clear_output(wait=True)

            print_peas()
            print_agent_path()
            print_initial_state(initial_env)
            print_current_state(env, mode="AGENT SIMPLE")
            print_history(history)
            print_result(env)
            break

        time.sleep(delay)

    if env["step"] >= max_steps and len(env["dirt"]) > 0:
        print_result(env)


# =========================================================
# RUN - MANUAL
# =========================================================

def run_manual(env):
    history = []
    initial_env = copy_environment(env)

    while True:
        clear_output(wait=True)

        percept = sensor(env)

        print_peas()
        print_initial_state(initial_env)
        print_current_state(env, mode="MANUAL")
        print_history(history)

        if percept["dirt_left"] == 0:
            print_result(env)
            break

        print("Nhập action: SUCK / UP / DOWN / LEFT / RIGHT / STOP")
        action = input("Action = ").upper().strip()

        if action == "STOP":
            print("Dừng MANUAL mode.")
            print_result(env)
            break

        if action not in ["SUCK", "UP", "DOWN", "LEFT", "RIGHT"]:
            history.append(f"Action '{action}' không tồn tại.")
            continue

        if action in ACTIONS and action not in percept["legal_actions"]:
            history.append(f"Action '{action}' không hợp lệ tại vị trí {percept['position']}.")
            continue

        old_pos = percept["position"]
        old_dirty = percept["dirty_here"]
        old_perf = performance(env)

        apply_action(env, action)

        new_pos = env["agent_pos"]
        new_perf = performance(env)

        history.append(
            f"Step {env['step']:02d} | Action: {action:<5} | "
            f"Position: {old_pos} -> {new_pos} | "
            f"Dirty before: {old_dirty} | "
            f"Performance: {old_perf}/{env['initial_dirt_count']} -> {new_perf}/{env['initial_dirt_count']}"
        )


# =========================================================
# MAIN
# =========================================================

def main():
    clear_output(wait=True)

    print("CHƯƠNG TRÌNH MÁY HÚT BỤI - SIMPLE AGENT")
    print("Môi trường: 3x3, không có vật cản")
    print()

    try:
        dirt_count = int(input("Nhập số ô bụi random từ 1 đến 9: "))
    except:
        dirt_count = 4

    if dirt_count < 1:
        dirt_count = 1

    if dirt_count > 9:
        dirt_count = 9

    env = random_environment(dirt_count)

    print()
    print("CHỌN CHẾ ĐỘ")
    print("1. MANUAL - Người chơi tự nhập action")
    print("2. AGENT SIMPLE - Agent tự chạy")
    choice = input("Nhập 1 hoặc 2: ").strip()

    if choice == "1":
        run_manual(env)

    elif choice == "2":
        run_agent_simple(
            env=env,
            max_steps=50,
            delay=0.4,
            animate=True
        )

    else:
        print("Bạn nhập sai chế độ.")


main()

PEAS CHO AGENT MÁY HÚT BỤI - KHÔNG CÓ VẬT CẢN
P - Performance:
  - Hút sạch bụi, số bước càng ít càng tốt

E - Environment:
  - Phòng 3x3, có ô sạch và ô bẩn, không có vật cản

A - Actuators:
  - SUCK, UP, DOWN, LEFT, RIGHT, STOP

S - Sensors:
  - Biết vị trí hiện tại, biết ô hiện tại có bụi không, biết số bụi còn lại, biết action hợp lệ

ĐƯỜNG ĐI CỐ ĐỊNH CỦA SIMPLE AGENT
(0,0) -> (0,1) -> (0,2)
                    ↓
(1,0) <- (1,1) <- (1,2)
↓
(2,0) -> (2,1) -> (2,2)

TRẠNG THÁI BAN ĐẦU
Step ban đầu   : 0
Performance    : 0 / 9
Agent position : (0, 0)
Số bụi ban đầu : 9

+------+------+------+
|  AD  |  D   |  D   |
+------+------+------+
|  D   |  D   |  D   |
+------+------+------+
|  D   |  D   |  D   |
+------+------+------+
A = Agent | D = Dirt | AD = Agent đứng trên bụi | . = Sạch

TRẠNG THÁI HIỆN TẠI
Mode           : AGENT SIMPLE
Step hiện tại  : 17
Performance    : 9 / 9
Agent position : (2, 2)
Dirty here     : False
Dirt left      : 0
Legal actions  : ['UP', 'LEFT']

+------+--